<a href="https://colab.research.google.com/github/DaniJonesOcean/etc-impacts-great-lakes/blob/main/notebooks/cartopy/fig1_glr_overview.ipynb/Figure_1_Map.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cartopy shapely pyproj

In [2]:
# Mounting Google Drive in Google Colab
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Rectangle

# ============================================================
# I/O paths (Google Drive)
# ============================================================
CSV_PATH = "/content/gdrive/MyDrive/ETC Clustering Paper/cfsr_storms_labeled_k2.csv"
OUTDIR  = "/content/gdrive/MyDrive/ETC Clustering Paper/figures"
os.makedirs(OUTDIR, exist_ok=True)

# Load data
df = pd.read_csv(CSV_PATH)

# ============================================================
# Figure 1 settings (match Figure 4 extent)
# ============================================================
lon_min, lon_max = -130, -66
lat_min, lat_max = 24, 65

# Placeholder GLR box (EDIT LATER once colleague sends bounds)
# Format: (lon_left, lat_bottom, width_deg, height_deg)
GLR_BOX = (-92.0, 41.0, 17.0, 8.5)

def setup_ax(ax):
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, alpha=0.25)
    ax.add_feature(cfeature.OCEAN, alpha=0.15)
    ax.add_feature(cfeature.LAKES, alpha=0.55)
    ax.add_feature(cfeature.RIVERS, linewidth=0.35, alpha=0.35)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)

    gl = ax.gridlines(
        draw_labels=True, dms=True, x_inline=False, y_inline=False,
        color="k", alpha=0.25, linestyle="--", linewidth=0.5
    )
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}

def smooth2d(H, iters=2):
    K = np.array([[1, 2, 1],
                  [2, 4, 2],
                  [1, 2, 1]], dtype=float)
    K /= K.sum()
    A = H.astype(float)
    for _ in range(iters):
        P = np.pad(A, 1, mode="edge")
        A = (
            K[0,0]*P[:-2,:-2] + K[0,1]*P[:-2,1:-1] + K[0,2]*P[:-2,2:] +
            K[1,0]*P[1:-1,:-2] + K[1,1]*P[1:-1,1:-1] + K[1,2]*P[1:-1,2:] +
            K[2,0]*P[2:,:-2] + K[2,1]*P[2:,1:-1] + K[2,2]*P[2:,2:]
        )
    return A

def plot_fig1_glr_overview(
    df,
    add_density=True,
    bins=120,
    smooth_iters=2,
    clip_q=0.98,
    outdir=OUTDIR,
    fname="fig1_glr_overview_density.png",
):
    # Validate columns
    required = {"lon_gen", "lat_gen"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in CSV: {sorted(missing)}")

    d = df.dropna(subset=["lon_gen", "lat_gen"]).copy()

    fig = plt.figure(figsize=(12, 7))
    ax = plt.axes(projection=ccrs.PlateCarree())
    setup_ax(ax)

    # Optional: all-storm genesis density background (both classes combined)
    if add_density:
        H, xedges, yedges = np.histogram2d(
            d["lon_gen"], d["lat_gen"],
            bins=[bins, bins],
            range=[[lon_min, lon_max], [lat_min, lat_max]],
        )
        Hs = smooth2d(H.T, iters=smooth_iters)

        if np.any(Hs > 0):
            upper = np.quantile(Hs[Hs > 0], clip_q)
            Hplot = np.clip(Hs, 0, upper)
        else:
            Hplot = Hs

        X, Y = np.meshgrid(xedges, yedges)
        pm = ax.pcolormesh(
            X, Y, Hplot,
            transform=ccrs.PlateCarree(),
            shading="auto",
            alpha=0.85,
        )
        cbar = fig.colorbar(pm, ax=ax, shrink=0.85, pad=0.02)
        cbar.set_label(f"All-storm genesis density (smoothed, clipped @ {int(clip_q*100)}th)")

    # GLR box overlay
    rect = Rectangle(
        (GLR_BOX[0], GLR_BOX[1]),
        GLR_BOX[2], GLR_BOX[3],
        fill=False, linewidth=2.5, edgecolor="k",
        transform=ccrs.PlateCarree(),
    )
    ax.add_patch(rect)
    ax.text(
        GLR_BOX[0] + 0.7,
        GLR_BOX[1] + GLR_BOX[3] - 0.8,
        "Great Lakes Region (GLR)",
        fontsize=12, weight="bold",
        transform=ccrs.PlateCarree(),
        bbox=dict(facecolor="white", alpha=0.65, edgecolor="none", pad=2.5)
    )

    ax.set_title("Figure 1: Study domain (GLR) within broader storm genesis climatology", fontsize=14)
    fig.tight_layout()

    outpath = os.path.join(outdir, fname)
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return outpath

# ---- Run and save Figure 1 ----
outpath = plot_fig1_glr_overview(df, add_density=True, fname="fig1_glr_overview_density.png")
print("Saved:", outpath)

# Optional: box-only version
# outpath2 = plot_fig1_glr_overview(df, add_density=False, fname="fig1_glr_overview_boxonly.png")
# print("Saved:", outpath2)


/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_ocean.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_rivers_lake_centerlines.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloadin

Saved: /content/gdrive/MyDrive/ETC Clustering Paper/figures/fig1_glr_overview_density.png
